# Model Deployment: Service & Gateway

**Session 4 — Deep Dive: Deployment, Inference & MLOps**  
**Notebook 1 of 3** | ~15 minutes

---

## What You Will Learn

| Concept | What We Cover |
|---------|---------------|
| **Model Service** | Deploy a registered model as a managed SPCS inference service |
| **Auto-Capture** | Automatically log every request/response for observability |
| **Snowflake Gateway** | Create a stable endpoint with traffic splitting capabilities |
| **Service Lifecycle** | Monitor, suspend, resume, and scale services |

---

## Architecture Overview

```
 ┌──────────────────────────────────────────────────────────────┐
 │                    Snowflake Model Serving                    │
 │                                                              │
 │  ┌─────────────┐      ┌─────────────────┐                   │
 │  │   Model      │      │  Inference       │                   │
 │  │   Registry   │─────▶│  Service (SPCS)  │                   │
 │  │              │      │  + Auto-Capture   │                   │
 │  └─────────────┘      └────────┬──────────┘                   │
 │                                 │                              │
 │                        ┌────────▼──────────┐                   │
 │                        │   Gateway          │                   │
 │                        │   (Stable URL)     │                   │
 │                        │   Traffic Split    │                   │
 │                        └────────┬──────────┘                   │
 │                                 │                              │
 │  ┌───────────┐  ┌───────────┐  │  ┌───────────────────┐      │
 │  │ SQL Query  │  │ REST API  │◀─┘  │ Inference Table   │      │
 │  │ (Warehouse)│  │ (External)│      │ (Auto-Captured)   │      │
 │  └───────────┘  └───────────┘      └───────────────────┘      │
 └──────────────────────────────────────────────────────────────┘
```

## 1 | Environment Setup

In [ ]:
%cd ..
%load_ext autoreload

In [ ]:
import os
import time

from snowflake.ml.registry import Registry

from utils import get_config
from utils import get_session

config = get_config("config.yaml")
DB = config.snowflake.database
SCHEMA = config.snowflake.schema_name
WH = config.snowflake.warehouse
COMPUTE_POOL = config.compute.compute_pool
MODEL_NAME = config.model.model_name
SERVICE_NAME = config.deploy.service_name

session = get_session(
    config.snowflake.connection_name,
    database_name=DB,
    schema_name=SCHEMA,
    warehouse_name=WH,
)
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")
print(f"Model: {MODEL_NAME} | Service: {SERVICE_NAME} | Pool: {COMPUTE_POOL}")

## 2 | Get Model Version from Registry

In [ ]:
reg = Registry(session=session, database_name=DB, schema_name=SCHEMA)

model = reg.get_model(MODEL_NAME)
mv = model.last()

print(f"Model: {MODEL_NAME}")
print(f"Latest version: {mv.version_name}")
print(f"\nAvailable functions:")
mv.show_functions()

## 3 | Deploy as Inference Service

We deploy the model as a managed SPCS service with:
- **`ingress_enabled=True`** — creates a public HTTP endpoint
- **`autocapture=True`** — logs every request/response to the inference table
- **`max_instances`** — controls horizontal scaling

> **Note:** `autocapture` is immutable — you cannot enable/disable it after service creation.

In [ ]:
# Drop the service if it already exists
session.sql(f"""
    DROP SERVICE IF EXISTS {DB}.{SCHEMA}.{SERVICE_NAME}
""").collect()

mv.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool=COMPUTE_POOL,
    ingress_enabled=True,
    max_instances=config.deploy.max_instances,
    autocapture=True,
)
print(f"Service '{SERVICE_NAME}' creation initiated.")
print("This typically takes 5-15 minutes for CPU models...")

## 4 | Monitor Service Status

In [ ]:
import time

for i in range(20):
    result = session.sql(f"DESCRIBE SERVICE {DB}.{SCHEMA}.{SERVICE_NAME}").collect()
    status = result[0]["status"] if result else "UNKNOWN"
    print(f"[{i + 1}] Service status: {status}")
    if status == "RUNNING":
        print("\nService is READY!")
        break
    time.sleep(60)
else:
    print("\nTimeout — check status manually with:")
    print(f"  DESCRIBE SERVICE {DB}.{SCHEMA}.{SERVICE_NAME};")

## 5 | Get Service Endpoints

In [ ]:
services_df = mv.list_services()
print("Deployed services:")
print(services_df[["name", "status", "inference_endpoint", "autocapture_enabled"]].to_string())

ingress_url = services_df[services_df["name"] == f"{DB}.{SCHEMA}.{SERVICE_NAME}"]["inference_endpoint"].iloc[0]
print(f"\nPublic endpoint: {ingress_url}")

In [ ]:
session.sql(f"SHOW ENDPOINTS IN SERVICE {DB}.{SCHEMA}.{SERVICE_NAME}").show()

## 6 | Create a Snowflake Gateway (Stable Endpoint)

The Gateway provides:
- **Stable URL** — permanent hostname that survives service recreation
- **Traffic splitting** — route % of traffic to different model versions (canary/blue-green)
- **Automatic failover** — redirects traffic away from unhealthy services

```
 Clients ──▶ Gateway (stable URL) ──▶ Service V1 (90%)
                                  └──▶ Service V2 (10%)
```

In [ ]:
GATEWAY_NAME = "PATIENT_RISK_GATEWAY"

gateway_sql = f"""
CREATE OR REPLACE GATEWAY {DB}.{SCHEMA}.{GATEWAY_NAME}
  FROM SPECIFICATION $$
    spec:
      type: traffic_split
      split_type: custom
      targets:
        - type: endpoint
          value: {DB}.{SCHEMA}.{SERVICE_NAME}!inference
          weight: 100
  $$;
"""

session.sql(gateway_sql).collect()
print(f"Gateway '{GATEWAY_NAME}' created with 100% traffic to {SERVICE_NAME}")

In [ ]:
gateway_info = session.sql(f"DESC GATEWAY {DB}.{SCHEMA}.{GATEWAY_NAME}").collect()
for row in gateway_info:
    print(row)

gateway_endpoint = session.sql(
    f'SELECT "ingress_url" FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))'
).collect()
print(f"\nGateway endpoint: {gateway_endpoint}")

## 7 | Grant Endpoint Access

By default only the service owner can access endpoints.  
Grant access to other roles for external REST API calls:

In [ ]:
session.sql(f"""
    GRANT SERVICE ROLE {DB}.{SCHEMA}.{SERVICE_NAME}!ALL_ENDPOINTS_USAGE
    TO ROLE SYSADMIN
""").collect()
print("Endpoint access granted to SYSADMIN.")

## 8 | Configure Auto-Suspend

Set the service to automatically suspend after a period of inactivity to save costs:

In [ ]:
session.sql(f"""
    ALTER SERVICE {DB}.{SCHEMA}.{SERVICE_NAME}
    SET AUTO_SUSPEND_SECS = {config.deploy.auto_suspend_secs}
""").collect()
print(
    f"Auto-suspend set to {config.deploy.auto_suspend_secs} seconds ({config.deploy.auto_suspend_secs // 60} minutes)."
)

## 9 | Service Management Reference

| Operation | Command |
|-----------|--------|
| Suspend | `ALTER SERVICE <name> SUSPEND;` |
| Resume | `ALTER SERVICE <name> RESUME;` |
| Scale | `ALTER SERVICE <name> SET MIN_INSTANCES=N, MAX_INSTANCES=M;` |
| Logs | `CALL SYSTEM$GET_SERVICE_LOGS('<name>', 0, 'model-inference');` |
| Drop | `DROP SERVICE <name>;` |
| Drop Gateway | `DROP GATEWAY <name>;` |

In [ ]:
# session.sql(f"ALTER SERVICE {DB}.{SCHEMA}.{SERVICE_NAME} SUSPEND").collect()
# session.sql(f"ALTER SERVICE {DB}.{SCHEMA}.{SERVICE_NAME} RESUME").collect()
# session.sql(f"DROP SERVICE {DB}.{SCHEMA}.{SERVICE_NAME}").collect()
# session.sql(f"DROP GATEWAY {DB}.{SCHEMA}.{GATEWAY_NAME}").collect()

## Summary

In this notebook we:

1. **Deployed** the model as a managed SPCS inference service with auto-capture enabled
2. **Created a Gateway** for a stable endpoint with traffic splitting capabilities
3. **Granted access** to roles for external REST API usage
4. **Configured auto-suspend** for cost management

---

**Next:** [02_inference.ipynb](./02_inference.ipynb) — Run real-time and batch inference via SQL and REST API